# TUGAS PERTEMUAN 12 DATA SCIENCE

**Nama Lengkap:** IKRAM  
**NIM:** 240401020139  
**Kelas:** IF 405

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh 3 transaksi teratas:', transaksi[:3])
print('Jumlah total transaksi:', len(transaksi))


Contoh 3 transaksi teratas: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah total transaksi: 50


In [2]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print('Tampilan 5 baris pertama hasil One-Hot Encoding:')
print(df.head())

Tampilan 5 baris pertama hasil One-Hot Encoding:
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [8]:
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print('\nTop 10 Frequent Itemset Teratas:')
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Top 10 Frequent Itemset Teratas:
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


In [9]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)

rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print('Daftar Aturan Asosiasi Terkuat (Berdasarkan Nilai Lift):')
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Daftar Aturan Asosiasi Terkuat (Berdasarkan Nilai Lift):
         antecedents consequents  support  confidence      lift
9        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
14  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
12      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
13     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
8      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
11     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
15   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


### Analisis Temuan Aturan Asosiasi (Market Basket Analysis)

* **Aturan Terkuat**: Berdasarkan hasil pengujian di atas, pola asosiasi antara **{Roti} -> {Selai}** (atau sebaliknya) menjadi aturan yang paling kuat karena memiliki nilai **Lift tertinggi** (di atas 1) dan nilai *confidence* yang besar.
* **Logika Bisnis**: Pola ini sangat masuk akal secara bisnis (*business sense*), karena di dunia nyata konsumen memang sering membeli roti dan selai secara bersamaan sebagai satu paket menu sarapan. Hasil ini membuktikan bahwa algoritma Apriori berhasil mendeteksi pola transaksi yang telah kita suntikkan di Langkah 1.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
                 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print("Rekomendasi Produk Serupa dengan 'Roti':", rekomendasi_serupa('Roti'))

Rekomendasi Produk Serupa dengan 'Roti': ['Selai', 'Sereal', 'Susu']


In [11]:
produk_target = 'Roti'

rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print('Rekomendasi dari Aturan Asosiasi (Apriori):')
print(rules_terkait[['consequents', 'lift']].head())

print('\nRekomendasi dari Content-Based Filtering:')
print(rekomendasi_serupa(produk_target))

Rekomendasi dari Aturan Asosiasi (Apriori):
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based Filtering:
['Selai', 'Sereal', 'Susu']


### Diskusi Hasil Perbandingan Metode Rekomendasi

* **Konsistensi Hasil**: Kedua metode memberikan hasil rekomendasi yang berbeda untuk produk **'Roti'**.
  * *Association Rules* merekomendasikan **'Selai'** karena berdasarkan pola transaksi nyata di mana kedua barang tersebut sering dibeli bersamaan.
  * *Content-Based Filtering* merekomendasikan produk sesama jenis kategori *Bakery* seperti **'Sereal'** atau jenis roti lainnya, tanpa peduli apakah produk tersebut pernah dibeli bersama atau tidak.
* **Strategi Penggunaan Terbaik**:
  * **Association Rules** sangat cocok dipakai untuk penataan posisi rak pajangan di toko fisik (*cross-selling*) atau pembuatan paket belanja hemat (*product bundling*).
  * **Content-Based Filtering** sangat bagus dipakai saat ada **produk baru** yang belum memiliki riwayat transaksi sama sekali (*cold start*), karena sistem cukup membaca karakteristik/kategori dari produk tersebut.
  * **Sistem Hybrid**: Untuk hasil yang paling maksimal di platform digital modern, menggabungkan kedua metode ini (*Hybrid*) adalah pilihan terbaik agar rekomendasi yang diberikan bisa tetap relevan sekaligus bervariasi.